<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/main/RA1_LAB2/EXPERIENCIA_1%20/RA2_Lab_N%C2%B02_Exp1_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiencia 1 — Sistema supervisado de detección de fraude bancario

**Dataset:** `creditcard.csv`

**Grupo:** N° 8 ·

**Integrantes:** Acosta Alex, Bareiro Santiago, Borges Agustín · **Fecha:** 04/2026



In [ ]:
import sys, subprocess
def _pip(pkgs):
    try:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",*pkgs],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception:
        try:
            subprocess.check_call([sys.executable,"-m","pip","install","-q",
                                   "--break-system-packages",*pkgs],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception:
            pass

try: import imblearn
except ImportError: _pip(["imbalanced-learn"])
try: import gdown
except ImportError: _pip(["gdown"])


In [ ]:
import os, sys

os.environ["TF_CPP_MIN_LOG_LEVEL"]   = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"]  = "0"
os.environ["GRPC_VERBOSITY"]         = "ERROR"
os.environ["GLOG_minloglevel"]       = "3"

def _silence_fd2():
    """Redirige el fd 2 (stderr a nivel C) a /dev/null."""
    try:
        sys.stderr.flush()
        saved = os.dup(2)
        devnull = os.open(os.devnull, os.O_WRONLY)
        os.dup2(devnull, 2)
        os.close(devnull)
        return saved
    except Exception:
        return None

def _restore_fd2(saved):
    if saved is None: return
    try:
        sys.stderr.flush()
        os.dup2(saved, 2)
        os.close(saved)
    except Exception:
        pass

_saved_fd = _silence_fd2()
try:
    import tensorflow as tf
    try:
        import absl.logging
        absl.logging.set_verbosity(absl.logging.ERROR)
    except Exception:
        pass
finally:
    _restore_fd2(_saved_fd)

import warnings, logging, random, time, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.get_logger().setLevel("ERROR")
tf.autograph.set_verbosity(0)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)
tf.random.set_seed(SEED)
try: tf.keras.utils.set_random_seed(SEED)
except Exception: pass

pd.set_option("display.max_columns", 50)
sns.set_theme(context="notebook", style="whitegrid", palette="deep")

FIG_DIR   = "figures"
FIG_DIR2  = "figs"          # D0: copia a 200 dpi con nombre exp1_<nombre>.png
os.makedirs(FIG_DIR,  exist_ok=True)
os.makedirs(FIG_DIR2, exist_ok=True)

FIGURAS_EXP1 = {}   # nombre_corto -> ruta en figs/

def _nombre_corto(name):
    """'fig07_comparacion_estrategias.png' -> 'comparacion_estrategias'"""
    base = os.path.splitext(os.path.basename(name))[0]
    partes = base.split("_", 1)
    return partes[1] if (len(partes) == 2 and partes[0].startswith("fig")) else base

def savefig(name, fig=None, dpi=140):
    """Guarda la figura en figures/ (dpi original) y en figs/exp1_<nombre>.png a 200 dpi."""
    fig = fig or plt.gcf()
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    corto = _nombre_corto(name)
    path2 = os.path.join(FIG_DIR2, f"exp1_{corto}.png")
    fig.savefig(path2, dpi=200, bbox_inches="tight", facecolor="white")
    FIGURAS_EXP1[corto] = path2
    return path

# --- D0/D1.2: cronometro global y acelerador detectado ---
T0_NOTEBOOK = time.time()
_gpus = tf.config.list_physical_devices("GPU")
ACELERADOR = (_gpus[0].name if _gpus else "CPU")
TIEMPOS = {}    # D1.2(a): nombre -> segundos

print("TensorFlow:", tf.__version__)
print("GPU:", "si" if _gpus else "no")
print("Acelerador detectado:", ACELERADOR)

## 1. Carga del dataset


In [ ]:
# Carga del dataset (local o Colab): si el CSV ya existe se usa tal cual,
# solo se descarga desde Drive cuando falta.
FILE_ID  = "1Zy1fV1U5nXj8XXhcqEI4YjEmlzbodk0n"
ZIP_PATH = "creditcard.zip"
CSV_PATH = "creditcard.csv"

if not os.path.exists(CSV_PATH):
    if not os.path.exists(ZIP_PATH):
        try:
            import gdown
            gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", ZIP_PATH, quiet=False)
        except Exception as e:
            raise FileNotFoundError(
                f"No se encontro '{CSV_PATH}' ni '{ZIP_PATH}' en {os.getcwd()} y la descarga "
                f"fallo ({e}). Copiar manualmente creditcard.csv junto al notebook.")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")
else:
    print(f"Usando el CSV local: {os.path.abspath(CSV_PATH)}")

df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}  |  Fraudes: {int(df['Class'].sum())}  |  Tasa: {df['Class'].mean():.4%}")
df.head()

In [ ]:
# Integridad y deduplicacion  ——  D1.2(b) tabla antes/despues  +  D1.2(c) verificacion de Time

# ---------- (c) Time es real? Se verifica ANTES de tocar el DataFrame ----------
time_ok_monotona   = bool(df["Time"].is_monotonic_increasing)
time_min           = float(df["Time"].min())
time_max           = float(df["Time"].max())
time_horas         = (time_max - time_min) / 3600.0
time_n_unicos      = int(df["Time"].nunique())
time_es_entera     = bool(np.allclose(df["Time"].values, np.round(df["Time"].values)))
time_delta_max     = float(df["Time"].diff().max())
time_delta_neg     = int((df["Time"].diff() < 0).sum())

print("=" * 74)
print("VERIFICACIÓN DE LA COLUMNA `Time`  (D1.2c)")
print("=" * 74)
print(f"  Origen del dato        : pd.read_csv('{CSV_PATH}')  (archivo descargado de Drive)")
print(f"  Monótona creciente     : {time_ok_monotona}   (violaciones: {time_delta_neg})")
print(f"  Rango [min, max]       : [{time_min:,.0f} s , {time_max:,.0f} s]")
print(f"  Duración cubierta      : {time_horas:,.2f} h  ({time_horas/24:.2f} días)")
print(f"  Valores enteros (seg)  : {time_es_entera}")
print(f"  Valores distintos      : {time_n_unicos:,}")
print(f"  Salto máximo entre filas: {time_delta_max:,.0f} s")
print("-" * 74)
if time_ok_monotona and time_es_entera and 40 <= time_horas <= 56:
    print("  CONCLUSIÓN: `Time` son SEGUNDOS REALES transcurridos desde la primera")
    print("  transacción, ordenados cronológicamente y cubriendo ~2 días completos.")
    print("  NO es una columna sintética ni reordenada: el análisis circadiano es VÁLIDO.")
else:
    print("  CONCLUSIÓN: revisar, alguna condición no se cumple (ver valores arriba).")
print("=" * 74)

# ---------- (b) Antes / despues de deduplicar ----------
n_antes       = int(len(df))
fraudes_antes = int(df["Class"].sum())
tasa_antes    = float(df["Class"].mean())
ratio_antes   = (n_antes - fraudes_antes) / max(fraudes_antes, 1)
n_nulos       = int(df.isna().sum().sum())
n_dups        = int(df.duplicated().sum())
dups_fraude   = int(df[df.duplicated()]["Class"].sum())

df = df.drop_duplicates().reset_index(drop=True)

n_despues       = int(len(df))
fraudes_despues = int(df["Class"].sum())
tasa_despues    = float(df["Class"].mean())
ratio_despues   = (n_despues - fraudes_despues) / max(fraudes_despues, 1)

tabla_dedup = pd.DataFrame({
    "N filas":        [n_antes, n_despues, n_antes - n_despues],
    "Fraudes":        [fraudes_antes, fraudes_despues, fraudes_antes - fraudes_despues],
    "Legítimas":      [n_antes - fraudes_antes, n_despues - fraudes_despues,
                       (n_antes - fraudes_antes) - (n_despues - fraudes_despues)],
    "Tasa fraude (%)":[100*tasa_antes, 100*tasa_despues, 100*(tasa_despues - tasa_antes)],
    "Ratio 1:x":      [ratio_antes, ratio_despues, ratio_despues - ratio_antes],
}, index=["ANTES de deduplicar", "DESPUÉS de deduplicar", "Diferencia"])

print("\nINTEGRIDAD Y DEDUPLICACIÓN  (D1.2b)")
print(f"  Nulos totales           : {n_nulos}")
print(f"  Filas duplicadas exactas: {n_dups:,}  (de las cuales fraude: {dups_fraude})")
print(tabla_dedup.round(4).to_string())
print(f"\n  >>> Cifras a citar en el informe y en la Figura 1: "
      f"N={n_despues:,} · fraudes={fraudes_despues} · tasa={100*tasa_despues:.4f}% · ratio 1:{ratio_despues:,.0f}")
print(f"  >>> Se eliminaron {n_antes - n_despues:,} duplicados. "
      f"TODA figura/tabla posterior usa el DataFrame DEDUPLICADO.")

## 2. EDA

In [ ]:
# fig01 — Desbalance de clases
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df["Class"].value_counts().sort_index()
axes[0].bar(["Legítima (0)", "Fraude (1)"], counts.values, color=["#1f77b4", "#d62728"])
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=10)
axes[0].set_yscale("log"); axes[0].set_ylabel("N (log)")
axes[0].set_title("Conteo por clase")

pct = counts / counts.sum() * 100
axes[1].pie(pct.values, labels=[f"Legítima\n{pct.iloc[0]:.3f}%", f"Fraude\n{pct.iloc[1]:.3f}%"],
            colors=["#1f77b4","#d62728"], startangle=90, wedgeprops={"edgecolor":"white"})
axes[1].set_title("Proporción relativa")
plt.tight_layout()
savefig("fig01_desbalance_clases.png")
plt.show()

ratio = counts.iloc[0] / counts.iloc[1]
print(f"Por cada fraude hay ~{ratio:,.0f} transacciones legítimas.")


In [ ]:
# fig02 — Distribución de Amount y Time (con ciclo de 24 h)
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0,0].hist(df["Amount"].clip(upper=df["Amount"].quantile(0.99)),
               bins=80, color="#1f77b4", edgecolor="white", linewidth=0.3)
axes[0,0].set_title("Amount (hasta p99)"); axes[0,0].set_yscale("log")
axes[0,0].set_xlabel("Monto"); axes[0,0].set_ylabel("Frecuencia (log)")

axes[0,1].hist(np.log1p(df["Amount"]), bins=80, color="#2ca02c", edgecolor="white", linewidth=0.3)
axes[0,1].set_title("log(1 + Amount)"); axes[0,1].set_xlabel("log(1 + Amount)")

hour = (df["Time"] / 3600.0) % 24
sns.histplot(x=hour[df["Class"]==0], bins=48, ax=axes[1,0], color="#1f77b4", stat="density", alpha=0.6)
axes[1,0].set_title("Hora del día — legítimas"); axes[1,0].set_xlabel("Hora (mod 24h)")
sns.histplot(x=hour[df["Class"]==1], bins=48, ax=axes[1,1], color="#d62728", stat="density", alpha=0.6)
axes[1,1].set_title("Hora del día — fraudes"); axes[1,1].set_xlabel("Hora (mod 24h)")

plt.tight_layout()
savefig("fig02_distribucion_amount_time.png")
plt.show()


In [ ]:
# fig03 — Correlación de V1..V28 y Amount con Class
v_cols = [f"V{i}" for i in range(1, 29)]
corrs = df[v_cols + ["Amount", "Class"]].corr()["Class"].drop("Class").sort_values()

fig, ax = plt.subplots(figsize=(11, 4.5))
colors = ["#d62728" if abs(c) > 0.1 else "#7f7f7f" for c in corrs.values]
ax.bar(corrs.index, corrs.values, color=colors)
ax.set_title("Correlación de Pearson con Class (fraude=1)")
ax.set_ylabel("Pearson r")
ax.axhline(0, color='black', lw=0.5)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
savefig("fig03_correlacion_features_class.png")
plt.show()

top_features = corrs.abs().sort_values(ascending=False).head(6).index.tolist()
print("Top 6 por |corr|:", top_features)


In [ ]:
# fig04 — Distribución por clase en las 6 features más informativas
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, top_features):
    sns.kdeplot(data=df[df["Class"]==0], x=col, ax=ax, fill=True, alpha=0.3, label="Legítima")
    sns.kdeplot(data=df[df["Class"]==1], x=col, ax=ax, fill=True, alpha=0.5,
                label="Fraude", color="#d62728")
    ax.set_title(col); ax.legend(fontsize=8)
plt.suptitle("Distribuciones condicionales — features más informativas", y=1.02)
plt.tight_layout()
savefig("fig04_distribuciones_top_features.png")
plt.show()


## 3. División train / val / test estratificada

70 / 15 / 15. Se hace antes de cualquier transformación supervisada y antes de cualquier resampling para evitar leakage.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class"]).copy()
y = df["Class"].astype(int).copy()

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15/0.85, random_state=SEED, stratify=y_trainval)

print(pd.DataFrame({
    "N": [len(y_train), len(y_val), len(y_test)],
    "Fraudes": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "Tasa (%)": [100*y_train.mean(), 100*y_val.mean(), 100*y_test.mean()],
}, index=["Train","Val","Test"]).round(4))


## 4. Pipeline de preprocesamiento

`RobustScaler` para `Amount` (cola larga), `StandardScaler` para `Time`, passthrough para `V1..V28` (ya escaladas por PCA).

In [ ]:
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ("amount", RobustScaler(),  ["Amount"]),
    ("time",   StandardScaler(), ["Time"]),
    ("v",      "passthrough",   v_cols),
])
preprocessor.fit(X_train)

X_train_p = preprocessor.transform(X_train)
X_val_p   = preprocessor.transform(X_val)
X_test_p  = preprocessor.transform(X_test)
print("Shape train preprocesado:", X_train_p.shape)


## 5. Función de evaluación común

In [ ]:
from sklearn.metrics import (precision_recall_curve, average_precision_score,
                             roc_auc_score, confusion_matrix,
                             fbeta_score, f1_score, precision_score, recall_score)

def evaluate(y_true, y_score, threshold=0.5, label=""):
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "model": label, "threshold": threshold,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "f2":        fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "pr_auc":    average_precision_score(y_true, y_score),
        "roc_auc":   roc_auc_score(y_true, y_score),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def print_eval(res):
    print(f"--- {res['model']}  (umbral={res['threshold']:.4f}) ---")
    print(f"  Precision={res['precision']:.4f}  Recall={res['recall']:.4f}  "
          f"F1={res['f1']:.4f}  F2={res['f2']:.4f}")
    print(f"  PR-AUC={res['pr_auc']:.4f}   ROC-AUC={res['roc_auc']:.4f}")
    print(f"  TN={res['tn']:,}  FP={res['fp']:,}  FN={res['fn']:,}  TP={res['tp']:,}")


## 6. Baseline no supervisado: Isolation Forest

Se reentrena IF sobre el mismo `train` de esta experiencia (199k filas, sin etiquetas) y se evalúa sobre el mismo `test` (~42k) que verá el supervisado. Esto aísla el efecto del paradigma: misma escala de datos, mismo split, misma métrica.

In [ ]:
from sklearn.ensemble import IsolationForest

iforest = IsolationForest(
    n_estimators=200,
    contamination=float(y_train.mean()),  # tasa real del train (~0.0017)
    max_samples="auto",
    random_state=SEED, n_jobs=-1,
)
iforest.fit(X_train_p)

# Score continuo (mayor = más anómalo) para evaluación por curva
val_score_if  = -iforest.decision_function(X_val_p)
test_score_if = -iforest.decision_function(X_test_p)

# Umbral derivado del contamination (top ~0.17% como anomalía)
thr_if = np.quantile(val_score_if, 1 - float(y_train.mean()))
res_if_val  = evaluate(y_val,  val_score_if,  threshold=thr_if, label="IF [VAL]")
res_if_test = evaluate(y_test, test_score_if, threshold=thr_if, label="IF [TEST]")
print_eval(res_if_val); print()
print_eval(res_if_test)


In [ ]:
# fig05 — Curva PR del Isolation Forest sobre test (línea base)
p, r, _ = precision_recall_curve(y_test, test_score_if)
ap = average_precision_score(y_test, test_score_if)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(r, p, color="#7f7f7f", lw=2, label=f"Isolation Forest (PR-AUC={ap:.3f})")
ax.axhline(y_test.mean(), ls="--", color="black", lw=0.8,
           label=f"Baseline aleatorio ({y_test.mean():.4f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Curva PR — Isolation Forest sobre test")
ax.legend(loc="lower left"); ax.grid(alpha=0.3)
plt.tight_layout()
savefig("fig05_isolation_forest_baseline.png")
plt.show()


## 7. Numero de muestras óptimo

Regresión logística con `class_weight='balanced'` sobre dataset completo vs submuestra estratificada (200 negativos por positivo). Mismo set de validación.

In [ ]:
from sklearn.linear_model import LogisticRegression

def stratified_subsample(X, y, neg_per_pos=200, random_state=SEED):
    rng = np.random.RandomState(random_state)
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_neg = min(len(neg_idx), neg_per_pos * len(pos_idx))
    neg_sample = rng.choice(neg_idx, size=n_neg, replace=False)
    keep = np.concatenate([pos_idx, neg_sample]); rng.shuffle(keep)
    return X[keep], (y.iloc[keep] if hasattr(y, "iloc") else y[keep])

X_tr_sub, y_tr_sub = stratified_subsample(X_train_p, y_train, neg_per_pos=200)

t0 = time.time()
lr_full = LogisticRegression(max_iter=2000, class_weight="balanced",
                             solver="liblinear", random_state=SEED).fit(X_train_p, y_train)
t_full = time.time() - t0

t0 = time.time()
lr_sub = LogisticRegression(max_iter=2000, class_weight="balanced",
                            solver="liblinear", random_state=SEED).fit(X_tr_sub, y_tr_sub)
t_sub = time.time() - t0

score_full = lr_full.predict_proba(X_val_p)[:, 1]
score_sub  = lr_sub.predict_proba(X_val_p)[:, 1]

res_full = evaluate(y_val, score_full, 0.5, f"LR completo (N={X_train_p.shape[0]:,}, {t_full:.1f}s)")
res_sub  = evaluate(y_val, score_sub,  0.5, f"LR submuestreado (N={X_tr_sub.shape[0]:,}, {t_sub:.1f}s)")
print_eval(res_full); print()
print_eval(res_sub)


In [ ]:
# fig06 — Curva PR completo vs submuestreado
fig, ax = plt.subplots(figsize=(7, 5))
for score, name, color in [(score_full, f"Completo N={X_train_p.shape[0]:,}", "#1f77b4"),
                           (score_sub,  f"Submuestra N={X_tr_sub.shape[0]:,}", "#ff7f0e")]:
    p, r, _ = precision_recall_curve(y_val, score)
    ap = average_precision_score(y_val, score)
    ax.plot(r, p, label=f"{name}  (PR-AUC={ap:.3f})", color=color, lw=2)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Curva PR (validación) — entrenar con todo vs con submuestra estratificada")
ax.legend(loc="lower left"); ax.grid(alpha=0.3)
plt.tight_layout()
savefig("fig06_muestreo_estratificado.png")
plt.show()


## 8. Comparación de estrategias de desbalance

Misma MLP base con tres estrategias: `class_weight`, SMOTE en pipeline (solo train) y `RandomUnderSampler`. Métrica de selección: PR-AUC sobre validación.

In [ ]:
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

def build_mlp(input_dim, hidden=(64, 32), dropout=0.3, lr=1e-3, name="mlp"):
    inp = Input(shape=(input_dim,), name="inp")
    x = inp
    for i, units in enumerate(hidden):
        x = layers.Dense(units, kernel_initializer="he_normal", name=f"dense_{i}")(x)
        x = layers.BatchNormalization(name=f"bn_{i}")(x)
        x = layers.Activation("relu", name=f"relu_{i}")(x)
        x = layers.Dropout(dropout, name=f"drop_{i}")(x)
    out = layers.Dense(1, activation="sigmoid", name="out")(x)
    m = Model(inp, out, name=name)
    m.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc"),
                       tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
                       tf.keras.metrics.Precision(name="prec"),
                       tf.keras.metrics.Recall(name="rec")])
    return m

def make_callbacks(patience=5):
    return [EarlyStopping(monitor="val_pr_auc", mode="max", patience=patience,
                          restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor="val_pr_auc", mode="max",
                              factor=0.5, patience=3, verbose=0)]

INPUT_DIM = X_train_p.shape[1]
EPOCHS, BATCH = 30, 2048


In [ ]:
# Estrategia 1 — class_weight   (+ D1.2a: instrumentación de tiempo)
from sklearn.utils.class_weight import compute_class_weight
cw_arr = compute_class_weight("balanced", classes=np.array([0,1]), y=y_train.values)
class_weight_dict = {0: float(cw_arr[0]), 1: float(cw_arr[1])}

tf.keras.utils.set_random_seed(SEED)
mlp_cw = build_mlp(INPUT_DIM, name="mlp_cw")
_t0 = time.time()
hist_cw = mlp_cw.fit(X_train_p, y_train.values,
                    validation_data=(X_val_p, y_val.values),
                    epochs=EPOCHS, batch_size=BATCH,
                    class_weight=class_weight_dict,
                    callbacks=make_callbacks(), verbose=0)
TIEMPOS["fit_class_weight_s"]      = time.time() - _t0
TIEMPOS["resample_class_weight_s"] = 0.0          # no hay resampling
epocas_cw = len(hist_cw.history["loss"])
print(f"[tiempo] class_weight: fit={TIEMPOS['fit_class_weight_s']:.1f}s en {epocas_cw} épocas "
      f"({TIEMPOS['fit_class_weight_s']/epocas_cw:.2f}s/época) · N_train={X_train_p.shape[0]:,}")

score_cw = mlp_cw.predict(X_val_p, verbose=0).ravel()
res_cw = evaluate(y_val, score_cw, 0.5, "MLP + class_weight [VAL]")
print_eval(res_cw)

In [ ]:
# Estrategia 2 — SMOTE (solo sobre train)   (+ D1.2a)
from imblearn.over_sampling import SMOTE
smote = SMOTE(sampling_strategy=0.1, random_state=SEED, k_neighbors=5)
_t0 = time.time()
X_tr_sm, y_tr_sm = smote.fit_resample(X_train_p, y_train)
TIEMPOS["resample_smote_s"] = time.time() - _t0
print(f"Tras SMOTE: N={len(y_tr_sm):,}  fraudes={int(y_tr_sm.sum())}  "
      f"(resampling: {TIEMPOS['resample_smote_s']:.1f}s)")

tf.keras.utils.set_random_seed(SEED)
mlp_sm = build_mlp(INPUT_DIM, name="mlp_smote")
_t0 = time.time()
hist_sm = mlp_sm.fit(X_tr_sm, y_tr_sm,
                    validation_data=(X_val_p, y_val.values),
                    epochs=EPOCHS, batch_size=BATCH,
                    callbacks=make_callbacks(), verbose=0)
TIEMPOS["fit_smote_s"] = time.time() - _t0
epocas_sm = len(hist_sm.history["loss"])
print(f"[tiempo] SMOTE: resample={TIEMPOS['resample_smote_s']:.1f}s + "
      f"fit={TIEMPOS['fit_smote_s']:.1f}s en {epocas_sm} épocas "
      f"({TIEMPOS['fit_smote_s']/epocas_sm:.2f}s/época) · N_train={len(y_tr_sm):,}")

score_sm = mlp_sm.predict(X_val_p, verbose=0).ravel()
res_sm = evaluate(y_val, score_sm, 0.5, "MLP + SMOTE [VAL]")
print_eval(res_sm)

In [ ]:
# Estrategia 3 — Undersampling   (+ D1.2a)
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(sampling_strategy=0.1, random_state=SEED)
_t0 = time.time()
X_tr_us, y_tr_us = rus.fit_resample(X_train_p, y_train)
TIEMPOS["resample_undersample_s"] = time.time() - _t0
print(f"Tras RUS: N={len(y_tr_us):,}  fraudes={int(y_tr_us.sum())}  "
      f"(resampling: {TIEMPOS['resample_undersample_s']:.1f}s)")

tf.keras.utils.set_random_seed(SEED)
mlp_us = build_mlp(INPUT_DIM, name="mlp_us")
_t0 = time.time()
hist_us = mlp_us.fit(X_tr_us, y_tr_us,
                    validation_data=(X_val_p, y_val.values),
                    epochs=EPOCHS, batch_size=BATCH,
                    callbacks=make_callbacks(), verbose=0)
TIEMPOS["fit_undersample_s"] = time.time() - _t0
epocas_us = len(hist_us.history["loss"])
print(f"[tiempo] Undersampling: resample={TIEMPOS['resample_undersample_s']:.1f}s + "
      f"fit={TIEMPOS['fit_undersample_s']:.1f}s en {epocas_us} épocas "
      f"({TIEMPOS['fit_undersample_s']/epocas_us:.2f}s/época) · N_train={len(y_tr_us):,}")

score_us = mlp_us.predict(X_val_p, verbose=0).ravel()
res_us = evaluate(y_val, score_us, 0.5, "MLP + Undersampling [VAL]")
print_eval(res_us)

In [ ]:
# fig07 — Comparación de estrategias en validación
comparison = pd.DataFrame([res_cw, res_sm, res_us])[
    ["model","precision","recall","f1","f2","pr_auc","roc_auc","tn","fp","fn","tp"]]
print(comparison.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
for score, name, color in [(score_cw, "class_weight", "#1f77b4"),
                           (score_sm, "SMOTE",        "#2ca02c"),
                           (score_us, "Undersample",  "#ff7f0e")]:
    p, r, _ = precision_recall_curve(y_val, score)
    ap = average_precision_score(y_val, score)
    ax.plot(r, p, label=f"{name}  (PR-AUC={ap:.3f})", color=color, lw=2)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Comparación de estrategias de desbalance (validación)")
ax.legend(loc="lower left"); ax.grid(alpha=0.3)
plt.tight_layout()
savefig("fig07_comparacion_estrategias_desbalance.png")
plt.show()

# Selección automática por PR-AUC
estrategias = {"class_weight": res_cw, "smote": res_sm, "undersample": res_us}
MEJOR = max(estrategias, key=lambda k: estrategias[k]["pr_auc"])
print(f"\nMejor estrategia por PR-AUC: {MEJOR}  (PR-AUC={estrategias[MEJOR]['pr_auc']:.4f})")

# --- D1.2a: tabla costo/beneficio de cada estrategia ---
tabla_tiempos_estrategias = pd.DataFrame({
    "N train":      [X_train_p.shape[0], len(y_tr_sm), len(y_tr_us)],
    "Fraudes train":[int(y_train.sum()), int(y_tr_sm.sum()), int(y_tr_us.sum())],
    "Resample (s)": [TIEMPOS["resample_class_weight_s"], TIEMPOS["resample_smote_s"],
                     TIEMPOS["resample_undersample_s"]],
    "Fit (s)":      [TIEMPOS["fit_class_weight_s"], TIEMPOS["fit_smote_s"],
                     TIEMPOS["fit_undersample_s"]],
    "Épocas":       [epocas_cw, epocas_sm, epocas_us],
    "s/época":      [TIEMPOS["fit_class_weight_s"]/epocas_cw,
                     TIEMPOS["fit_smote_s"]/epocas_sm,
                     TIEMPOS["fit_undersample_s"]/epocas_us],
    "PR-AUC val":   [res_cw["pr_auc"], res_sm["pr_auc"], res_us["pr_auc"]],
}, index=["class_weight", "SMOTE", "Undersample"])
tabla_tiempos_estrategias["Total (s)"] = (tabla_tiempos_estrategias["Resample (s)"]
                                          + tabla_tiempos_estrategias["Fit (s)"])
print("\nCOSTO COMPUTACIONAL POR ESTRATEGIA  (D1.2a) - acelerador:", ACELERADOR)
print(tabla_tiempos_estrategias.round(4).to_string())

## 9. MLP final con la mejor estrategia

In [ ]:
tf.keras.utils.set_random_seed(SEED)
mlp_final = build_mlp(INPUT_DIM, hidden=(64,32), dropout=0.3, lr=1e-3, name="mlp_final")
mlp_final.summary()

# D1.2a: se guarda la referencia a los callbacks para leer la época restaurada
cbs_final = make_callbacks(patience=8)
es_final  = cbs_final[0]          # EarlyStopping(monitor="val_pr_auc", restore_best_weights=True)

fit_kw = dict(validation_data=(X_val_p, y_val.values),
              epochs=50, batch_size=BATCH,
              callbacks=cbs_final, verbose=0)

_t0 = time.time()
if MEJOR == "class_weight":
    hist_final = mlp_final.fit(X_train_p, y_train.values, class_weight=class_weight_dict, **fit_kw)
elif MEJOR == "smote":
    hist_final = mlp_final.fit(X_tr_sm, y_tr_sm, **fit_kw)
else:
    hist_final = mlp_final.fit(X_tr_us, y_tr_us, **fit_kw)
TIEMPOS["fit_mlp_final_s"] = time.time() - _t0

epocas_final = len(hist_final.history["loss"])

# Época en la que EarlyStopping restauró los pesos (1-indexada).
# Se prefiere el atributo del callback; si la versión de Keras no lo expone,
# se reconstruye como el argmax de val_pr_auc (la métrica monitoreada, mode="max").
_best_from_hist = int(np.argmax(hist_final.history["val_pr_auc"])) + 1
_best_attr = getattr(es_final, "best_epoch", None)
if _best_attr is None:
    epoca_restaurada = _best_from_hist
    _fuente_epoca = "argmax(val_pr_auc) [callback sin atributo best_epoch]"
else:
    epoca_restaurada = int(_best_attr) + 1     # Keras la guarda 0-indexada
    _fuente_epoca = "EarlyStopping.best_epoch"

print(f"Estrategia usada       : {MEJOR}")
print(f"Épocas efectivas       : {epocas_final}  (tope=50, patience=8)")
print(f"Parámetros del modelo  : {mlp_final.count_params():,}")
print(f"[tiempo] fit MLP final : {TIEMPOS['fit_mlp_final_s']:.1f}s "
      f"({TIEMPOS['fit_mlp_final_s']/epocas_final:.2f}s/época) · acelerador={ACELERADOR}")
print(f"EarlyStopping restauró los pesos de la ÉPOCA {epoca_restaurada}/{epocas_final} "
      f"(val_pr_auc={max(hist_final.history['val_pr_auc']):.4f}) [fuente: {_fuente_epoca}]")
print(f"Épocas desperdiciadas tras el óptimo: {epocas_final - epoca_restaurada}")

In [ ]:
# D1.2a — Latencia de inferencia por transacción (batch=1)
#   Se miden 2 caminos: llamada directa al grafo (lo que usaría un servicio online)
#   y model.predict() (API de alto nivel, con overhead de tf.data).
_x1 = X_test_p[:1].astype("float32")

N_WARMUP, N_PASADAS = 30, 200

for _ in range(N_WARMUP):                       # calentamiento (traza el grafo, aloca en GPU)
    _ = mlp_final(_x1, training=False)

_lat = []
for _ in range(N_PASADAS):
    _t0 = time.perf_counter()
    _ = mlp_final(_x1, training=False)
    _lat.append((time.perf_counter() - _t0) * 1000.0)   # ms
_lat = np.array(_lat)

for _ in range(5):
    _ = mlp_final.predict(_x1, verbose=0)
_latp = []
for _ in range(N_PASADAS):
    _t0 = time.perf_counter()
    _ = mlp_final.predict(_x1, verbose=0)
    _latp.append((time.perf_counter() - _t0) * 1000.0)
_latp = np.array(_latp)

# Throughput en lote (referencia de batch scoring sobre todo el test)
_t0 = time.perf_counter()
_ = mlp_final.predict(X_test_p, batch_size=BATCH, verbose=0)
_t_batch = time.perf_counter() - _t0

LAT = {
    "latencia_media_ms":  float(_lat.mean()),
    "latencia_mediana_ms":float(np.median(_lat)),
    "latencia_p95_ms":    float(np.percentile(_lat, 95)),
    "latencia_p99_ms":    float(np.percentile(_lat, 99)),
    "latencia_std_ms":    float(_lat.std()),
    "latencia_predict_media_ms": float(_latp.mean()),
    "throughput_batch_tx_por_s": float(X_test_p.shape[0] / _t_batch),
    "batch_test_total_s": float(_t_batch),
}
TIEMPOS["inferencia_batch_test_s"] = _t_batch

print(f"LATENCIA DE INFERENCIA - batch=1, {N_WARMUP} de calentamiento + {N_PASADAS} pasadas "
      f"· acelerador={ACELERADOR}")
print(f"  Llamada directa mlp_final(x): media={LAT['latencia_media_ms']:.3f} ms · "
      f"mediana={LAT['latencia_mediana_ms']:.3f} ms · p95={LAT['latencia_p95_ms']:.3f} ms · "
      f"p99={LAT['latencia_p99_ms']:.3f} ms · sd={LAT['latencia_std_ms']:.3f} ms")
print(f"  API model.predict()         : media={LAT['latencia_predict_media_ms']:.3f} ms")
print(f"  Scoring por lotes del test  : {X_test_p.shape[0]:,} tx en {_t_batch:.2f}s "
      f"-> {LAT['throughput_batch_tx_por_s']:,.0f} tx/s")
print(f"  Lectura: con una latencia mediana de {LAT['latencia_mediana_ms']:.2f} ms por transacción, "
      f"el scoring es despreciable frente al presupuesto típico de autorización (~100 ms).")

In [ ]:
# fig08 — Curvas de aprendizaje del modelo final
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_final.history["loss"], label="Train", color="#1f77b4")
axes[0].plot(hist_final.history["val_loss"], label="Val", color="#d62728")
axes[0].set_title("Pérdida"); axes[0].set_xlabel("Época"); axes[0].set_ylabel("BCE")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist_final.history["pr_auc"], label="Train", color="#1f77b4")
axes[1].plot(hist_final.history["val_pr_auc"], label="Val", color="#d62728")
axes[1].set_title("PR-AUC"); axes[1].set_xlabel("Época"); axes[1].set_ylabel("PR-AUC")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
savefig("fig08_curvas_aprendizaje_mlp.png")
plt.show()


## 10. Análisis de umbral sobre validación

In [ ]:
score_val = mlp_final.predict(X_val_p, verbose=0).ravel()
prec_v, rec_v, thr_v = precision_recall_curve(y_val, score_val)
ap_v = average_precision_score(y_val, score_val)

eps = 1e-12
f1_grid = 2*prec_v*rec_v / (prec_v+rec_v+eps)
beta = 2.0
f2_grid = (1+beta**2)*prec_v*rec_v / (beta**2*prec_v + rec_v + eps)

idx_f1 = int(np.argmax(f1_grid[:-1]))
idx_f2 = int(np.argmax(f2_grid[:-1]))
thr_f1 = float(thr_v[idx_f1])
thr_f2 = float(thr_v[idx_f2])

print(f"PR-AUC validación: {ap_v:.4f}")
print(f"Umbral Max F1: {thr_f1:.4f}  -> P={prec_v[idx_f1]:.3f} R={rec_v[idx_f1]:.3f} F1={f1_grid[idx_f1]:.3f}")
print(f"Umbral Max F2: {thr_f2:.4f}  -> P={prec_v[idx_f2]:.3f} R={rec_v[idx_f2]:.3f} F2={f2_grid[idx_f2]:.3f}")


In [ ]:
# fig09 — Curva PR de validación con puntos óptimos y métricas vs umbral
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rec_v, prec_v, color="#1f77b4", lw=2, label=f"MLP final (PR-AUC={ap_v:.3f})")
axes[0].scatter(rec_v[idx_f1], prec_v[idx_f1], color="#2ca02c", s=140, zorder=5, marker="o",
                label=f"Max F1 @ thr={thr_f1:.4f}")
axes[0].scatter(rec_v[idx_f2], prec_v[idx_f2], color="#d62728", s=60, zorder=6, marker="x",
                label=f"Max F2 @ thr={thr_f2:.4f}")
axes[0].axhline(y_val.mean(), ls="--", color="gray", lw=1, label=f"Aleatorio ({y_val.mean():.4f})")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("PR (validación) con puntos óptimos")
axes[0].legend(loc="lower left"); axes[0].grid(alpha=0.3)

axes[1].plot(thr_v, prec_v[:-1], label="Precision", color="#1f77b4")
axes[1].plot(thr_v, rec_v[:-1],  label="Recall",    color="#d62728")
axes[1].plot(thr_v, f1_grid[:-1], label="F1",       color="#2ca02c", ls="--")
axes[1].plot(thr_v, f2_grid[:-1], label="F2",       color="#9467bd", ls="--")
axes[1].axvline(thr_f2, color="#9467bd", lw=0.8, alpha=0.5)
axes[1].set_xlabel("Umbral"); axes[1].set_ylabel("Valor")
axes[1].set_title("Métricas vs umbral"); axes[1].set_xscale("log")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
savefig("fig09_curva_pr_validacion_umbrales.png")
plt.show()

# ======================================================================
# D1.2(d) — Coinciden el umbral de máx-F1 y el de máx-F2?
# ======================================================================
UMBRALES_COINCIDEN = bool(np.isclose(thr_f1, thr_f2, rtol=0, atol=1e-9)) or (idx_f1 == idx_f2)
print("=" * 78)
print("COINCIDENCIA DE LOS UMBRALES ÓPTIMOS  (D1.2d)")
print("=" * 78)
print(f"  Umbral máx-F1 : {thr_f1:.4f}   (índice {idx_f1} de la grilla PR)")
print(f"  Umbral máx-F2 : {thr_f2:.4f}   (índice {idx_f2} de la grilla PR)")
print(f"  Diferencia    : {abs(thr_f1 - thr_f2):.3e}")
if UMBRALES_COINCIDEN:
    print(f"  >>> COINCIDEN: ambos criterios eligen EXACTAMENTE el mismo umbral ({thr_f2:.4f}).")
    print("  >>> Implicación: en esta curva PR NO hay trade-off entre F1 y F2, el mismo punto")
    print("      maximiza ambos, así que la elección de beta=2 no cambió el punto de operación.")
    print("      La preferencia por recall debe justificarse con el análisis de costo, no con F2.")
else:
    print("  >>> NO coinciden: F2 desplaza el punto de operación hacia mayor recall (esperado).")
print("=" * 78)

# ======================================================================
# D1.2(d) — Barrido de umbrales sobre VALIDACIÓN con lectura de costo
# ======================================================================
# Umbral de alto recall: el MAYOR umbral que aún alcanza recall >= 0.90 en validación
RECALL_OBJETIVO = 0.90
_mask_hr = rec_v[:-1] >= RECALL_OBJETIVO
if _mask_hr.any():
    idx_hr = int(np.max(np.where(_mask_hr)[0]))
    thr_hr = float(thr_v[idx_hr])
else:                                   # fallback: el de mayor recall disponible
    idx_hr = int(np.argmax(rec_v[:-1])); thr_hr = float(thr_v[idx_hr])
    print(f"[aviso] Ningún umbral alcanza recall>={RECALL_OBJETIVO}; se usa el de recall máximo.")

_umbrales = [
    (0.5,     "0.5 (por defecto)"),
    (thr_f1,  f"{thr_f1:.4f} (máx F1)"),
    (thr_f2,  f"{thr_f2:.4f} (máx F2)"),
    (thr_hr,  f"{thr_hr:.4f} (alto recall >={RECALL_OBJETIVO:.0%})"),
]

_filas = []
N_VAL       = int(len(y_val))
FRAUDES_VAL = int(y_val.sum())
LEGIT_VAL   = N_VAL - FRAUDES_VAL
for thr, nombre in _umbrales:
    r = evaluate(y_val, score_val, threshold=thr, label=nombre)
    _filas.append({
        "Umbral": nombre,
        "thr": thr,
        "Precision": r["precision"], "Recall": r["recall"],
        "F1": r["f1"], "F2": r["f2"],
        "TP": r["tp"], "FN": r["fn"], "FP": r["fp"], "TN": r["tn"],
        "Fraudes detectados": f"{r['tp']}/{FRAUDES_VAL}",
        "Alertas totales": r["tp"] + r["fp"],
        "FP por 10k legítimas": 1e4 * r["fp"] / LEGIT_VAL,
        "Revisiones por fraude detectado": (r["tp"] + r["fp"]) / max(r["tp"], 1),
    })

barrido_umbrales_val = pd.DataFrame(_filas)
print(f"\nBARRIDO DE UMBRALES SOBRE VALIDACIÓN "
      f"(N={N_VAL:,} · fraudes={FRAUDES_VAL} · legítimas={LEGIT_VAL:,})")
print(barrido_umbrales_val.drop(columns=["thr"]).round(4).to_string(index=False))

print("\nLECTURA EN COSTO DE NEGOCIO")
print("  FN = fraude que pasa -> pérdida directa del monto + costo de contracargo.")
print("  FP = transacción legítima bloqueada -> fricción con el cliente y costo de revisión manual.")
for f in _filas:
    print(f"  · thr={f['Umbral']}: se detectan {f['Fraudes detectados']} fraudes "
          f"({f['FN']} se escapan) a cambio de {f['FP']} falsos positivos sobre {LEGIT_VAL:,} "
          f"legítimas ({f['FP por 10k legítimas']:.2f} FP por cada 10.000) "
          f"-> {f['Revisiones por fraude detectado']:.2f} revisiones manuales por fraude detectado.")
print(f"\n  >>> Punto de operación elegido (thr={thr_f2:.4f}): es el mismo que maximiza F1, "
      f"y sostiene una tasa de falsos positivos casi nula sin sacrificar recall.")

## 11. Evaluación final sobre test — supervisado vs no supervisado

Umbral elegido en validación (F2). Test set intocado hasta este punto.

In [ ]:
score_test = mlp_final.predict(X_test_p, verbose=0).ravel()

# Reportes finales
finales = []
finales.append(res_if_test)  # ya calculado en sección 6
finales.append(evaluate(y_test, score_test, threshold=thr_f2,
                         label=f"MLP (umbral F2={thr_f2:.4f}) [TEST]"))
finales.append(evaluate(y_test, score_test, threshold=0.5,
                         label="MLP (umbral=0.5) [TEST]"))

for r in finales: print_eval(r); print()


In [ ]:
# Tabla final
tabla = pd.DataFrame(finales)[
    ["model","threshold","precision","recall","f1","f2","pr_auc","roc_auc","tn","fp","fn","tp"]
]
print(tabla.round(4).to_string(index=False))


In [ ]:
# fig10 — Curvas PR comparadas sobre test
fig, ax = plt.subplots(figsize=(8, 6))
for score, name, color in [(test_score_if, "Isolation Forest (no supervisado)", "#7f7f7f"),
                            (score_test,    "MLP supervisado",                  "#1f77b4")]:
    p, r, _ = precision_recall_curve(y_test, score)
    ap = average_precision_score(y_test, score)
    ax.plot(r, p, label=f"{name}  (PR-AUC={ap:.3f})", lw=2, color=color)
ax.axhline(y_test.mean(), ls="--", color="black", lw=0.8,
           label=f"Aleatorio ({y_test.mean():.4f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Curva PR sobre TEST — supervisado vs no supervisado")
ax.legend(loc="lower left"); ax.grid(alpha=0.3)
plt.tight_layout()
savefig("fig10_curva_pr_final_test.png")
plt.show()


In [ ]:
# fig11 — Matriz de confusión del modelo elegido sobre test
res_final = finales[1]   # MLP con umbral F2
cm = np.array([[res_final["tn"], res_final["fp"]],
               [res_final["fn"], res_final["tp"]]])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False,
            xticklabels=["Pred: Legítima","Pred: Fraude"],
            yticklabels=["Real: Legítima","Real: Fraude"], ax=ax)
ax.set_title(f"Matriz de confusión — MLP final (umbral={thr_f2:.4f})")
plt.tight_layout()
savefig("fig11_matriz_confusion_final.png")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"De {tp+fn} fraudes reales en test:")
print(f"  Detectados: {tp}  ({tp/(tp+fn)*100:.1f}%)")
print(f"  Perdidos  : {fn}  ({fn/(tp+fn)*100:.1f}%)")
print(f"De {tn+fp} legítimas en test:")
print(f"  Bloqueadas erróneamente: {fp}  ({fp/(tn+fp)*100:.3f}%)")


## 12. Arquitectura alternativa: Autoencoder semi-supervisado

Enfoque **semi-supervisado**: el autoencoder se entrena **únicamente con transacciones legítimas**
de `train` para aprender la variedad de la normalidad. La hipótesis es que un fraude, al no
pertenecer a esa variedad, se reconstruye peor (error de reconstrucción mayor) y que la
representación latente de 8 dimensiones concentra la señal discriminante.

- **Encoder:** 30 → 32 → 16 → 8 (ReLU)
- **Decoder:** 8 → 16 → 32 → 30 (salida lineal)
- **Pérdida:** MSE de reconstrucción · **EarlyStopping** sobre `val_loss` con `restore_best_weights`
- Luego el encoder se **congela**, se proyectan train/val/test al espacio latente de 8D y se entrena
  una `LogisticRegression(class_weight='balanced')` sobre esas representaciones.
- Evaluación con la **misma función `evaluate()`** que el resto de los modelos, sobre el **mismo** test.

In [ ]:
# 12.1 — Autoencoder entrenado SOLO con transacciones legítimas de train
from tensorflow.keras.callbacks import EarlyStopping as _ES

AE_LATENT = 8
mask_legit_train = (y_train.values == 0)
mask_legit_val   = (y_val.values   == 0)
X_train_legit = X_train_p[mask_legit_train].astype("float32")
X_val_legit   = X_val_p[mask_legit_val].astype("float32")

print(f"Train del AE (solo legítimas): {X_train_legit.shape[0]:,} de {X_train_p.shape[0]:,} "
      f"({int(y_train.sum())} fraudes EXCLUIDOS)")
print(f"Val del AE   (solo legítimas): {X_val_legit.shape[0]:,} de {X_val_p.shape[0]:,} "
      f"({int(y_val.sum())} fraudes EXCLUIDOS)")

def build_autoencoder(input_dim=INPUT_DIM, latent=AE_LATENT, lr=1e-3):
    """Encoder 30-32-16-8 (ReLU) y Decoder 8-16-32-30 (salida lineal)."""
    inp = Input(shape=(input_dim,), name="ae_in")
    e = layers.Dense(32, activation="relu", kernel_initializer="he_normal", name="enc_32")(inp)
    e = layers.Dense(16, activation="relu", kernel_initializer="he_normal", name="enc_16")(e)
    z = layers.Dense(latent, activation="relu", kernel_initializer="he_normal", name="latent")(e)
    d = layers.Dense(16, activation="relu", kernel_initializer="he_normal", name="dec_16")(z)
    d = layers.Dense(32, activation="relu", kernel_initializer="he_normal", name="dec_32")(d)
    out = layers.Dense(input_dim, activation=None, name="ae_out")(d)      # lineal
    ae  = Model(inp, out,  name="autoencoder")
    enc = Model(inp, z,    name="encoder")
    ae.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return ae, enc

tf.keras.utils.set_random_seed(SEED)
autoencoder, encoder = build_autoencoder()
autoencoder.summary()

ae_es = _ES(monitor="val_loss", mode="min", patience=10,
            restore_best_weights=True, verbose=0)

_t0 = time.time()
hist_ae = autoencoder.fit(
    X_train_legit, X_train_legit,                       # reconstrucción: X -> X
    validation_data=(X_val_legit, X_val_legit),         # val_loss = MSE solo sobre legítimas
    epochs=100, batch_size=BATCH,
    callbacks=[ae_es], verbose=0)
TIEMPOS["fit_autoencoder_s"] = time.time() - _t0

epocas_ae = len(hist_ae.history["loss"])
_best_ae = getattr(ae_es, "best_epoch", None)
epoca_restaurada_ae = (int(_best_ae) + 1 if _best_ae is not None
                       else int(np.argmin(hist_ae.history["val_loss"])) + 1)

print(f"[tiempo] fit autoencoder: {TIEMPOS['fit_autoencoder_s']:.1f}s en {epocas_ae} épocas "
      f"({TIEMPOS['fit_autoencoder_s']/epocas_ae:.2f}s/época)")
print(f"EarlyStopping (val_loss) restauró la ÉPOCA {epoca_restaurada_ae}/{epocas_ae} "
      f"· val_loss mínimo = {min(hist_ae.history['val_loss']):.6f}")

In [ ]:
# 12.2 — Error de reconstrucción medio por clase (D1.1)
def error_reconstruccion(X):
    X = X.astype("float32")
    rec = autoencoder.predict(X, batch_size=BATCH, verbose=0)
    return np.mean(np.square(X - rec), axis=1)      # MSE por transacción

err_train_ae = error_reconstruccion(X_train_p)
err_val_ae   = error_reconstruccion(X_val_p)
err_test_ae  = error_reconstruccion(X_test_p)

def _resumen_err(err, y, nombre):
    y = np.asarray(y)
    e0, e1 = err[y == 0], err[y == 1]
    return {
        "Partición": nombre,
        "MSE medio legítimas": float(e0.mean()),
        "MSE mediano legítimas": float(np.median(e0)),
        "MSE medio fraudes": float(e1.mean()),
        "MSE mediano fraudes": float(np.median(e1)),
        "Ratio medio (fraude/legítima)": float(e1.mean() / e0.mean()),
        "Ratio mediano": float(np.median(e1) / np.median(e0)),
    }

tabla_error_reconstruccion = pd.DataFrame([
    _resumen_err(err_train_ae, y_train.values, "Train"),
    _resumen_err(err_val_ae,   y_val.values,   "Val"),
    _resumen_err(err_test_ae,  y_test.values,  "Test"),
])
print("ERROR DE RECONSTRUCCIÓN MEDIO POR CLASE  (D1.1)")
print(tabla_error_reconstruccion.round(6).to_string(index=False))

# El error de reconstrucción es en sí mismo un score no supervisado: se evalúa como tal
res_ae_recon_test = evaluate(y_test, err_test_ae, threshold=float(np.quantile(err_val_ae, 1 - float(y_train.mean()))),
                             label="Autoencoder (error de reconstrucción) [TEST]")
print()
print_eval(res_ae_recon_test)

ae_ratio_test = float(tabla_error_reconstruccion.loc[
    tabla_error_reconstruccion["Partición"] == "Test", "Ratio medio (fraude/legítima)"].iloc[0])
print(f"\n  >>> En test, un fraude se reconstruye en promedio {ae_ratio_test:.2f}x peor que una "
      f"transacción legítima. Ese contraste, aprendido SIN ver un solo fraude, es el argumento "
      f"conceptual del enfoque semi-supervisado.")

# fig12 — Distribución del error de reconstrucción por clase (test)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
_e0, _e1 = err_test_ae[y_test.values == 0], err_test_ae[y_test.values == 1]
_bins = np.logspace(np.log10(max(err_test_ae.min(), 1e-6)), np.log10(err_test_ae.max()), 60)
axes[0].hist(_e0, bins=_bins, color="#1f77b4", alpha=0.6, density=True, label="Legítima")
axes[0].hist(_e1, bins=_bins, color="#d62728", alpha=0.6, density=True, label="Fraude")
axes[0].set_xscale("log"); axes[0].set_xlabel("MSE de reconstrucción (log)")
axes[0].set_ylabel("Densidad"); axes[0].legend()
axes[0].set_title("Error de reconstrucción por clase (test)")

axes[1].boxplot([_e0, _e1], showfliers=False)
axes[1].set_xticks([1, 2]); axes[1].set_xticklabels(["Legítima", "Fraude"])
axes[1].set_yscale("log"); axes[1].set_ylabel("MSE de reconstrucción (log)")
axes[1].set_title(f"Contraste de medianas (ratio medio = {ae_ratio_test:.2f}x)")
axes[1].grid(alpha=0.3)
plt.tight_layout()
savefig("fig12_autoencoder_error_reconstruccion.png")
plt.show()

In [ ]:
# 12.3 — Encoder CONGELADO + LogisticRegression(class_weight='balanced') sobre el latente 8D
encoder.trainable = False
for _l in encoder.layers:
    _l.trainable = False
print(f"Encoder congelado: {int(np.sum([np.prod(w.shape) for w in encoder.trainable_weights]))} "
      f"parámetros entrenables (debe ser 0)")

Z_train = encoder.predict(X_train_p.astype("float32"), batch_size=BATCH, verbose=0)
Z_val   = encoder.predict(X_val_p.astype("float32"),   batch_size=BATCH, verbose=0)
Z_test  = encoder.predict(X_test_p.astype("float32"),  batch_size=BATCH, verbose=0)
print(f"Proyección latente: {X_train_p.shape[1]}D -> {Z_train.shape[1]}D  "
      f"(train {Z_train.shape}, val {Z_val.shape}, test {Z_test.shape})")

_t0 = time.time()
lr_ae = LogisticRegression(max_iter=2000, class_weight="balanced",
                           solver="liblinear", random_state=SEED).fit(Z_train, y_train)
TIEMPOS["fit_lr_latente_s"] = time.time() - _t0
print(f"[tiempo] LogisticRegression sobre latente 8D: {TIEMPOS['fit_lr_latente_s']:.2f}s")

score_val_ae  = lr_ae.predict_proba(Z_val)[:, 1]
score_test_ae = lr_ae.predict_proba(Z_test)[:, 1]

# Umbral elegido en VALIDACIÓN con el mismo criterio (máx F2) usado para el MLP
prec_ae, rec_ae, thr_ae_grid = precision_recall_curve(y_val, score_val_ae)
_f2_ae = (1 + 2.0**2) * prec_ae * rec_ae / (2.0**2 * prec_ae + rec_ae + 1e-12)
idx_f2_ae = int(np.argmax(_f2_ae[:-1]))
thr_f2_ae = float(thr_ae_grid[idx_f2_ae])
print(f"Umbral máx-F2 del AE+LR en validación: {thr_f2_ae:.4f}")

res_ae_val  = evaluate(y_val,  score_val_ae,  threshold=thr_f2_ae,
                       label=f"AE(8D)+LR (umbral F2={thr_f2_ae:.4f}) [VAL]")
res_ae_test = evaluate(y_test, score_test_ae, threshold=thr_f2_ae,
                       label=f"AE(8D)+LR (umbral F2={thr_f2_ae:.4f}) [TEST]")
print(); print_eval(res_ae_val)
print();  print_eval(res_ae_test)

In [ ]:
# 12.4 — Tabla comparativa de TEST ampliada y curva PR superpuesta (D1.1)
#        Amplía `finales` (celda 36) y sustituye a la tabla de la celda 37 y a fig10.
finales_ext = list(finales) + [res_ae_recon_test, res_ae_test]

tabla_final_ext = pd.DataFrame(finales_ext)[
    ["model","threshold","precision","recall","f1","f2","pr_auc","roc_auc","tn","fp","fn","tp"]
]
print("TABLA COMPARATIVA FINAL SOBRE TEST (incluye el autoencoder)")
print(tabla_final_ext.round(4).to_string(index=False))

# fig13 — Curvas PR superpuestas sobre test: IF, MLP, AE(recon), AE(8D)+LR
fig, ax = plt.subplots(figsize=(8.5, 6))
for score, name, color, ls in [
        (test_score_if, "Isolation Forest (no supervisado)", "#7f7f7f", "-"),
        (err_test_ae,   "Autoencoder — error de reconstrucción", "#9467bd", "--"),
        (score_test_ae, "Autoencoder(8D) + LogisticRegression", "#2ca02c", "-"),
        (score_test,    "MLP supervisado",                   "#1f77b4", "-")]:
    p, r, _ = precision_recall_curve(y_test, score)
    ap = average_precision_score(y_test, score)
    ax.plot(r, p, label=f"{name}  (PR-AUC={ap:.3f})", lw=2, color=color, ls=ls)
ax.axhline(y_test.mean(), ls=":", color="black", lw=0.9,
           label=f"Aleatorio ({y_test.mean():.4f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Curvas PR sobre TEST — supervisado vs no supervisado vs semi-supervisado")
ax.legend(loc="upper right", fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
savefig("fig13_curva_pr_test_con_autoencoder.png")
plt.show()

print(f"\nRESUMEN - PR-AUC sobre test")
for _r in finales_ext:
    print(f"  {_r['model']:<58s} PR-AUC={_r['pr_auc']:.4f}  R={_r['recall']:.4f}  P={_r['precision']:.4f}")

In [ ]:
# ===== Reporte final del entorno + serialización de métricas (DIRECTIVA 0) =====
import sklearn, imblearn, json, platform, glob, shutil
import scipy, matplotlib, seaborn as _sns

T_TOTAL = time.time() - T0_NOTEBOOK

print(f"Python      : {sys.version.split()[0]}")
print(f"Platform    : {platform.platform()}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"Imbalanced  : {imblearn.__version__}")
print(f"TensorFlow  : {tf.__version__}")
print(f"Semilla     : {SEED}")
print(f"Acelerador  : {ACELERADOR}")
print(f"Tiempo total: {T_TOTAL:.1f}s ({T_TOTAL/60:.2f} min)")

def _r(x, nd=4):
    """Redondea floats a 4 decimales; deja ints/bools/str intactos."""
    if isinstance(x, (bool, np.bool_)):  return bool(x)
    if isinstance(x, (int, np.integer)): return int(x)
    if isinstance(x, (float, np.floating)):
        v = float(x)
        return v if not np.isfinite(v) else round(v, nd)
    return x

M = {}   # dict PLANO nombre -> valor

# ---------- entorno, semilla, acelerador, tiempo total ----------
M["entorno_python"]        = sys.version.split()[0]
M["entorno_platform"]      = platform.platform()
M["entorno_numpy"]         = np.__version__
M["entorno_pandas"]        = pd.__version__
M["entorno_sklearn"]       = sklearn.__version__
M["entorno_imblearn"]      = imblearn.__version__
M["entorno_tensorflow"]    = tf.__version__
M["entorno_scipy"]         = scipy.__version__
M["entorno_matplotlib"]    = matplotlib.__version__
M["entorno_seaborn"]       = _sns.__version__
M["semilla"]               = int(SEED)
M["acelerador"]            = str(ACELERADOR)
M["tiempo_total_s"]        = _r(T_TOTAL)
M["tiempo_total_min"]      = _r(T_TOTAL / 60.0)

# ---------- dataset y deduplicación (D1.2b) ----------
M["n_filas_antes_dedup"]      = int(n_antes)
M["n_fraudes_antes_dedup"]    = int(fraudes_antes)
M["tasa_fraude_antes_pct"]    = _r(100 * tasa_antes)
M["ratio_antes_dedup"]        = _r(ratio_antes)
M["n_duplicados_eliminados"]  = int(n_dups)
M["n_filas_despues_dedup"]    = int(n_despues)
M["n_fraudes_despues_dedup"]  = int(fraudes_despues)
M["tasa_fraude_despues_pct"]  = _r(100 * tasa_despues)
M["ratio_despues_dedup"]      = _r(ratio_despues)
M["n_nulos"]                  = int(n_nulos)

# ---------- Time real (D1.2c) ----------
M["time_monotona_creciente"] = bool(time_ok_monotona)
M["time_min_s"]              = _r(time_min)
M["time_max_s"]              = _r(time_max)
M["time_duracion_horas"]     = _r(time_horas)
M["time_valores_enteros"]    = bool(time_es_entera)
M["time_valores_unicos"]     = int(time_n_unicos)

# ---------- particiones ----------
M["n_train"]        = int(len(y_train)); M["fraudes_train"] = int(y_train.sum())
M["n_val"]          = int(len(y_val));   M["fraudes_val"]   = int(y_val.sum())
M["n_test"]         = int(len(y_test));  M["fraudes_test"]  = int(y_test.sum())
M["tasa_train_pct"] = _r(100 * float(y_train.mean()))
M["tasa_val_pct"]   = _r(100 * float(y_val.mean()))
M["tasa_test_pct"]  = _r(100 * float(y_test.mean()))
M["input_dim"]      = int(INPUT_DIM)

# ---------- métricas de cada modelo (aplanadas desde los dicts de evaluate()) ----------
_CLAVES = ["threshold","precision","recall","f1","f2","pr_auc","roc_auc","tn","fp","fn","tp"]
def _volcar(prefijo, res):
    for k in _CLAVES:
        M[f"{prefijo}_{k}"] = _r(res[k])
    M[f"{prefijo}_label"] = res["model"]

_volcar("if_val",        res_if_val)
_volcar("if_test",       res_if_test)
_volcar("lr_completo_val",     res_full)
_volcar("lr_submuestreado_val", res_sub)
_volcar("mlp_cw_val",    res_cw)
_volcar("mlp_smote_val", res_sm)
_volcar("mlp_us_val",    res_us)
_volcar("mlp_test_f2",   finales[1])
_volcar("mlp_test_05",   finales[2])
_volcar("ae_recon_test", res_ae_recon_test)
_volcar("ae_lr_val",     res_ae_val)
_volcar("ae_lr_test",    res_ae_test)

# ---------- selección de estrategia y umbrales (D1.2d) ----------
M["mejor_estrategia"]      = str(MEJOR)
M["pr_auc_val_mlp_final"]  = _r(ap_v)
M["umbral_max_f1"]         = _r(thr_f1)
M["umbral_max_f2"]         = _r(thr_f2)
M["umbrales_f1_f2_coinciden"] = bool(UMBRALES_COINCIDEN)
M["umbral_alto_recall"]    = _r(thr_hr)
_SLUGS = ["05", "maxf1", "maxf2", "altorecall"]   # mismo orden que `_umbrales` en la celda 34
for _i, (_, _f) in enumerate(barrido_umbrales_val.iterrows()):
    _slug = _SLUGS[_i]
    M[f"barrido_val_thr{_slug}_recall"] = _r(_f["Recall"])
    M[f"barrido_val_thr{_slug}_precision"] = _r(_f["Precision"])
    M[f"barrido_val_thr{_slug}_tp"] = int(_f["TP"])
    M[f"barrido_val_thr{_slug}_fn"] = int(_f["FN"])
    M[f"barrido_val_thr{_slug}_fp"] = int(_f["FP"])

# ---------- tiempos e inferencia (D1.2a) ----------
for _k, _v in TIEMPOS.items():
    M[f"tiempo_{_k}"] = _r(_v)
M["epocas_class_weight"]   = int(epocas_cw)
M["epocas_smote"]          = int(epocas_sm)
M["epocas_undersample"]    = int(epocas_us)
M["epocas_mlp_final"]      = int(epocas_final)
M["epoca_restaurada_early_stopping"] = int(epoca_restaurada)
M["mlp_final_parametros"]  = int(mlp_final.count_params())
M["n_train_smote"]         = int(len(y_tr_sm))
M["n_train_undersample"]   = int(len(y_tr_us))
M["n_train_submuestra_lr"] = int(X_tr_sub.shape[0])
M["tiempo_lr_completo_s"]  = _r(t_full)
M["tiempo_lr_submuestra_s"]= _r(t_sub)
for _k, _v in LAT.items():
    M[_k] = _r(_v)
M["latencia_n_pasadas"]    = int(N_PASADAS)
M["latencia_n_warmup"]     = int(N_WARMUP)

# ---------- autoencoder (D1.1) ----------
M["ae_latent_dim"]              = int(AE_LATENT)
M["ae_n_train_legitimas"]       = int(X_train_legit.shape[0])
M["ae_parametros"]              = int(autoencoder.count_params())
M["ae_epocas"]                  = int(epocas_ae)
M["ae_epoca_restaurada"]        = int(epoca_restaurada_ae)
M["ae_val_loss_min_mse"]        = _r(min(hist_ae.history["val_loss"]), 6)
M["ae_umbral_max_f2"]           = _r(thr_f2_ae)
for _, _f in tabla_error_reconstruccion.iterrows():
    _p = _f["Partición"].lower()
    M[f"ae_mse_medio_legitimas_{_p}"] = _r(_f["MSE medio legítimas"], 6)
    M[f"ae_mse_medio_fraudes_{_p}"]   = _r(_f["MSE medio fraudes"], 6)
    M[f"ae_ratio_medio_{_p}"]         = _r(_f["Ratio medio (fraude/legítima)"])

# ---------- figuras ----------
os.makedirs("figs", exist_ok=True)
for _n, _p in sorted(FIGURAS_EXP1.items()):
    M[f"figura_{_n}"] = _p
M["n_figuras_guardadas"] = int(len(FIGURAS_EXP1))

with open("metricas_exp1.json", "w", encoding="utf-8") as f:
    json.dump(M, f, ensure_ascii=False, indent=2, sort_keys=True)

print(f"\n>>> metricas_exp1.json escrito con {len(M)} claves.")
print(f">>> Figuras en figs/ (200 dpi, bbox_inches='tight'): {len(FIGURAS_EXP1)}")
for _p in sorted(glob.glob("figs/exp1_*.png")):
    print("   ", _p, f"({os.path.getsize(_p)/1024:.0f} KB)")

# --- Empaquetado portable de resultados (reemplaza a !zip; funciona en Windows) ---
try:
    _zip = shutil.make_archive("resultados_exp1", "zip", ".", "figs")
    print(f">>> Empaquetado portable: {_zip}")
except Exception as _e:
    print(f"[aviso] No se pudo empaquetar figs/: {_e}")

# --- Descarga automática solo si el notebook corre en Google Colab ---
try:
    from google.colab import files as _colab_files
    _colab_files.download("metricas_exp1.json")
except Exception:
    print(">>> Entorno local: metricas_exp1.json y resultados_exp1.zip quedan en", os.getcwd())